# Diffusion Model Experiments — MNIST & Fashion MNIST

This notebook trains DDPM diffusion models on MNIST and Fashion MNIST using a GPU runtime on Google Colab.

**Before running:** Go to `Runtime → Change runtime type → GPU`.

## 1. Setup

In [ ]:
# Mount Google Drive (optional — for saving outputs)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q pytorch-fid pyyaml matplotlib

In [ ]:
# Upload or clone your assignment4 code into Colab.
# Option A: Upload a zip and unzip it
# from google.colab import files
# uploaded = files.upload()  # upload assignment4.zip
# !unzip -qo assignment4.zip -d /content/assignment4

# Option B: Clone from a private repo (uncomment and edit)
# !git clone https://<TOKEN>@github.com/<user>/dl-labs.git /content/dl-labs

# Option C: If already mounted via Drive
# !cp -r /content/drive/MyDrive/dl-labs/assignment4 /content/assignment4

# Set the working directory to assignment4
import os
# Adjust this path to wherever your assignment4 code lives:
ASSIGNMENT_DIR = '/content/assignment4'
os.chdir(ASSIGNMENT_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Verify GPU is available
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU detected! Go to Runtime → Change runtime type → GPU')

In [ ]:
# Download datasets
!python download_data.py

## 2. Train Diffusion on MNIST

In [ ]:
!python train.py --config_file configs/config_diffusion_mnist.yaml --output_dir outputs/diffusion_mnist

In [ ]:
# Display forward & reverse diffusion visualizations for MNIST
import glob
from IPython.display import display, Image as IPImage

output_dir = 'outputs/diffusion_mnist'

print('=== MNIST Forward Diffusion ===')
for f in sorted(glob.glob(f'{output_dir}/*forward*.png')):
    print(f)
    display(IPImage(filename=f, width=800))

print('\n=== MNIST Reverse Diffusion (Generated Samples) ===')
for f in sorted(glob.glob(f'{output_dir}/*reverse*.png')):
    print(f)
    display(IPImage(filename=f, width=800))

## 3. Generate MNIST Images & Compute FID

In [ ]:
!python generate_images.py --config_file configs/config_diffusion_mnist.yaml --output_dir outputs/diffusion_mnist

In [ ]:
# Display generated grid
from IPython.display import display, Image as IPImage
display(IPImage(filename='outputs/diffusion_mnist/grid.png', width=600))

In [ ]:
# Compute FID score for MNIST
!python -m pytorch_fid outputs/diffusion_mnist/images tests/assets/fid-stats-fashion.npz --device cuda

## 4. Train Diffusion on Fashion MNIST

In [ ]:
!python train.py --config_file configs/config_diffusion_fashion.yaml --output_dir outputs/diffusion_fashion

In [ ]:
# Display forward & reverse diffusion visualizations for Fashion MNIST
import glob
from IPython.display import display, Image as IPImage

output_dir = 'outputs/diffusion_fashion'

print('=== Fashion MNIST Forward Diffusion ===')
for f in sorted(glob.glob(f'{output_dir}/*forward*.png')):
    print(f)
    display(IPImage(filename=f, width=800))

print('\n=== Fashion MNIST Reverse Diffusion (Generated Samples) ===')
for f in sorted(glob.glob(f'{output_dir}/*reverse*.png')):
    print(f)
    display(IPImage(filename=f, width=800))

## 5. Generate Fashion MNIST Images & Compute FID

In [ ]:
!python generate_images.py --config_file configs/config_diffusion_fashion.yaml --output_dir outputs/diffusion_fashion

In [ ]:
# Display generated grid
from IPython.display import display, Image as IPImage
display(IPImage(filename='outputs/diffusion_fashion/grid.png', width=600))

In [ ]:
# Compute FID score for Fashion MNIST
!python -m pytorch_fid outputs/diffusion_fashion/images tests/assets/fid-stats-fashion.npz --device cuda

## 6. (Optional) Copy outputs to Google Drive

In [ ]:
# Save outputs to Drive so they persist after Colab disconnects
import shutil
drive_dst = '/content/drive/MyDrive/assignment4_outputs'
os.makedirs(drive_dst, exist_ok=True)
for subdir in ['diffusion_mnist', 'diffusion_fashion']:
    src = f'outputs/{subdir}'
    dst = f'{drive_dst}/{subdir}'
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Copied {src} → {dst}')